# Raw IMDb Dataset Analysis

**Purpose.** Characterise the *full* official IMDb non-commercial dataset
(~10M titles, ~14M persons, ~85M principal credits) so the migrated CineExplorer
schema can be designed against the *full* design space — not just the 2,500-title
slice we currently work with.

This complements `docs/tsv_inspection.md` (which analyses only the filtered slice).
Findings here drive `docs/raw_imdb_analysis.md` and feed the Phase-2 ERD investigation
(`docs/erd_investigation.md`).

Tooling: **polars** for the heavy lifting (lazy / streaming, much smaller memory
footprint than pandas on the 85M-row `title.principals` file).

IMDb file specs: <https://developer.imdb.com/non-commercial-datasets/>

## Setup

In [ ]:
import json
import os
import sys
from pathlib import Path

import polars as pl

pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_cols(20)

NB_DIR = Path(__file__).parent if "__file__" in globals() else Path.cwd()
RAW = NB_DIR / ".." / "database" / "sources" / "imdb-official" / "raw"
OUT = NB_DIR / "output"
OUT.mkdir(exist_ok=True)

# IMDb's null marker is the literal two-character string "\N"
NULL = ["\\N"]

print("polars:", pl.__version__)
print("RAW dir:", RAW.resolve())
print("OUT dir:", OUT.resolve())

polars: 1.40.1
RAW dir: /home/duvu/All/82_Master_Uliege/_Y2Q2/KRR/info9014-krr-project/database/sources/imdb-official/raw
OUT dir: /home/duvu/All/82_Master_Uliege/_Y2Q2/KRR/info9014-krr-project/notebooks/output


## File overview

In [ ]:
files = [
    "title.basics.tsv.gz",
    "title.akas.tsv.gz",
    "title.crew.tsv.gz",
    "title.episode.tsv.gz",
    "title.principals.tsv.gz",
    "title.ratings.tsv.gz",
    "name.basics.tsv.gz",
]

overview = []
for f in files:
    p = RAW / f
    if p.exists():
        overview.append({
            "file": f,
            "size_mb": round(p.stat().st_size / 1024**2, 1),
        })
    else:
        overview.append({"file": f, "size_mb": None})

overview_df = pl.DataFrame(overview)
print(overview_df)

shape: (7, 2)
┌─────────────────────────┬─────────┐
│ file                    ┆ size_mb │
│ ---                     ┆ ---     │
│ str                     ┆ f64     │
╞═════════════════════════╪═════════╡
│ title.basics.tsv.gz     ┆ 211.0   │
│ title.akas.tsv.gz       ┆ 471.6   │
│ title.crew.tsv.gz       ┆ 77.6    │
│ title.episode.tsv.gz    ┆ 50.8    │
│ title.principals.tsv.gz ┆ 729.0   │
│ title.ratings.tsv.gz    ┆ 8.0     │
│ name.basics.tsv.gz      ┆ 288.8   │
└─────────────────────────┴─────────┘


## Helper: NULL / cardinality / distribution

In [ ]:
def null_rates(lf: pl.LazyFrame) -> pl.DataFrame:
    """Compute null rate per column (treating IMDb's '\\N' as null is handled at read time)."""
    schema = lf.collect_schema()
    cols = schema.names()
    total = lf.select(pl.len()).collect().item()
    out = []
    for c in cols:
        n_null = lf.select(pl.col(c).is_null().sum()).collect().item()
        out.append({"column": c, "null": n_null, "null_pct": round(100 * n_null / total, 2)})
    return pl.DataFrame(out)


def csv_cardinality(lf: pl.LazyFrame, col: str) -> pl.DataFrame:
    """Count distribution of comma-separated values per row in a single column."""
    return (
        lf.select(
            pl.col(col).str.split(",").list.len().alias("items_per_row")
        )
        .group_by("items_per_row")
        .len()
        .sort("items_per_row")
        .collect()
    )

## 1. `title.basics.tsv.gz` — title metadata

Schema: `tconst, titleType, primaryTitle, originalTitle, isAdult, startYear, endYear, runtimeMinutes, genres`

In [ ]:
basics = pl.scan_csv(
    RAW / "title.basics.tsv.gz",
    separator="\t",
    null_values=NULL,
    schema_overrides={
        "isAdult": pl.Int8,
        "startYear": pl.Int32,
        "endYear": pl.Int32,
        "runtimeMinutes": pl.Int32,
    },
    quote_char=None,  # IMDb does not quote fields
)

basics_total = basics.select(pl.len()).collect().item()
print(f"Total rows: {basics_total:,}")

Total rows: 12,478,987


In [ ]:
print("=== First 10 rows ===")
print(basics.head(10).collect())

=== First 10 rows ===
shape: (10, 9)
┌───────────┬───────────┬──────────┬──────────┬─────────┬──────────┬─────────┬──────────┬──────────┐
│ tconst    ┆ titleType ┆ primaryT ┆ original ┆ isAdult ┆ startYea ┆ endYear ┆ runtimeM ┆ genres   │
│ ---       ┆ ---       ┆ itle     ┆ Title    ┆ ---     ┆ r        ┆ ---     ┆ inutes   ┆ ---      │
│ str       ┆ str       ┆ ---      ┆ ---      ┆ i8      ┆ ---      ┆ i32     ┆ ---      ┆ str      │
│           ┆           ┆ str      ┆ str      ┆         ┆ i32      ┆         ┆ i32      ┆          │
╞═══════════╪═══════════╪══════════╪══════════╪═════════╪══════════╪═════════╪══════════╪══════════╡
│ tt0000001 ┆ short     ┆ Carmenci ┆ Carmenci ┆ 0       ┆ 1894     ┆ null    ┆ 1        ┆ Document │
│           ┆           ┆ ta       ┆ ta       ┆         ┆          ┆         ┆          ┆ ary,Shor │
│           ┆           ┆          ┆          ┆         ┆          ┆         ┆          ┆ t        │
│ tt0000002 ┆ short     ┆ Le clown ┆ Le clown ┆ 0     

In [ ]:
print("=== titleType distribution (full IMDb) ===")
titletype_dist = (
    basics.group_by("titleType")
    .len()
    .sort("len", descending=True)
    .collect()
)
print(titletype_dist)
titletype_dist.write_csv(OUT / "01_titletype_distribution.csv")

=== titleType distribution (full IMDb) ===
shape: (11, 2)
┌──────────────┬─────────┐
│ titleType    ┆ len     │
│ ---          ┆ ---     │
│ str          ┆ u32     │
╞══════════════╪═════════╡
│ tvEpisode    ┆ 9637883 │
│ short        ┆ 1128967 │
│ movie        ┆ 745163  │
│ video        ┆ 325414  │
│ tvSeries     ┆ 298867  │
│ tvMovie      ┆ 154790  │
│ tvMiniSeries ┆ 69997   │
│ tvSpecial    ┆ 57910   │
│ videoGame    ┆ 49008   │
│ tvShort      ┆ 10987   │
│ tvPilot      ┆ 1       │
└──────────────┴─────────┘


In [ ]:
print("=== NULL rates ===")
basics_nulls = null_rates(basics)
print(basics_nulls)
basics_nulls.write_csv(OUT / "02_basics_null_rates.csv")

=== NULL rates ===
shape: (9, 3)
┌────────────────┬──────────┬──────────┐
│ column         ┆ null     ┆ null_pct │
│ ---            ┆ ---      ┆ ---      │
│ str            ┆ i64      ┆ f64      │
╞════════════════╪══════════╪══════════╡
│ tconst         ┆ 0        ┆ 0.0      │
│ titleType      ┆ 0        ┆ 0.0      │
│ primaryTitle   ┆ 0        ┆ 0.0      │
│ originalTitle  ┆ 0        ┆ 0.0      │
│ isAdult        ┆ 0        ┆ 0.0      │
│ startYear      ┆ 1464907  ┆ 11.74    │
│ endYear        ┆ 12322598 ┆ 98.75    │
│ runtimeMinutes ┆ 7996549  ┆ 64.08    │
│ genres         ┆ 535747   ┆ 4.29     │
└────────────────┴──────────┴──────────┘


In [ ]:
print("=== isAdult distribution ===")
adult_dist = basics.group_by("isAdult").len().sort("isAdult").collect()
print(adult_dist)

=== isAdult distribution ===


In [ ]:
print("=== startYear range ===")
year_stats = basics.select([
    pl.col("startYear").min().alias("min"),
    pl.col("startYear").max().alias("max"),
    pl.col("startYear").is_null().sum().alias("null_count"),
]).collect()
print(year_stats)

=== startYear range ===
shape: (1, 3)
┌──────┬──────┬────────────┐
│ min  ┆ max  ┆ null_count │
│ ---  ┆ ---  ┆ ---        │
│ i32  ┆ i32  ┆ u32        │
╞══════╪══════╪════════════╡
│ 1874 ┆ 2115 ┆ 1464907    │
└──────┴──────┴────────────┘


In [ ]:
print("=== Genres: distinct values across full IMDb ===")
genres_distinct = (
    basics.select(pl.col("genres").str.split(",").alias("g"))
    .explode("g")
    .filter(pl.col("g").is_not_null())
    .group_by("g")
    .len()
    .sort("len", descending=True)
    .collect()
)
print(f"Distinct genres: {genres_distinct.height}")
print(genres_distinct)
genres_distinct.write_csv(OUT / "03_distinct_genres.csv")

=== Genres: distinct values across full IMDb ===
Distinct genres: 28
shape: (28, 2)
┌─────────────┬─────────┐
│ g           ┆ len     │
│ ---         ┆ ---     │
│ str         ┆ u32     │
╞═════════════╪═════════╡
│ Drama       ┆ 3482552 │
│ Comedy      ┆ 2407401 │
│ Talk-Show   ┆ 1550033 │
│ Short       ┆ 1324418 │
│ News        ┆ 1257417 │
│ Documentary ┆ 1187166 │
│ Romance     ┆ 1148792 │
│ Family      ┆ 915307  │
│ Reality-TV  ┆ 701059  │
│ Animation   ┆ 623395  │
│ Crime       ┆ 527324  │
│ Action      ┆ 526635  │
│ Adventure   ┆ 484833  │
│ Game-Show   ┆ 478483  │
│ Music       ┆ 451823  │
│ Adult       ┆ 407237  │
│ Sport       ┆ 325253  │
│ Fantasy     ┆ 280567  │
│ Horror      ┆ 269981  │
│ Mystery     ┆ 269820  │
│ Thriller    ┆ 207039  │
│ History     ┆ 183644  │
│ Biography   ┆ 131795  │
│ Sci-Fi      ┆ 126503  │
│ Musical     ┆ 99172   │
│ War         ┆ 42674   │
│ Western     ┆ 31576   │
│ Film-Noir   ┆ 876     │
└─────────────┴─────────┘


In [ ]:
print("=== Genres per title cardinality (max items in CSV) ===")
genres_card = csv_cardinality(basics.filter(pl.col("genres").is_not_null()), "genres")
print(genres_card)

=== Genres per title cardinality (max items in CSV) ===
shape: (3, 2)
┌───────────────┬─────────┐
│ items_per_row ┆ len     │
│ ---           ┆ ---     │
│ u32           ┆ u32     │
╞═══════════════╪═════════╡
│ 1             ┆ 6561631 │
│ 2             ┆ 3263683 │
│ 3             ┆ 2117926 │
└───────────────┴─────────┘


## 2. `title.akas.tsv.gz` — alternative / localized titles

Schema: `titleId, ordering, title, region, language, types, attributes, isOriginalTitle`

In [ ]:
akas = pl.scan_csv(
    RAW / "title.akas.tsv.gz",
    separator="\t",
    null_values=NULL,
    schema_overrides={"ordering": pl.Int32, "isOriginalTitle": pl.Int8},
    quote_char=None,
)
akas_total = akas.select(pl.len()).collect().item()
print(f"Total rows: {akas_total:,}")

Total rows: 57,013,962


In [ ]:
print("=== First 10 rows ===")
print(akas.head(10).collect())

=== First 10 rows ===
shape: (10, 8)
┌───────────┬──────────┬─────────────┬────────┬──────────┬─────────────┬─────────────┬─────────────┐
│ titleId   ┆ ordering ┆ title       ┆ region ┆ language ┆ types       ┆ attributes  ┆ isOriginalT │
│ ---       ┆ ---      ┆ ---         ┆ ---    ┆ ---      ┆ ---         ┆ ---         ┆ itle        │
│ str       ┆ i32      ┆ str         ┆ str    ┆ str      ┆ str         ┆ str         ┆ ---         │
│           ┆          ┆             ┆        ┆          ┆             ┆             ┆ i8          │
╞═══════════╪══════════╪═════════════╪════════╪══════════╪═════════════╪═════════════╪═════════════╡
│ tt0000001 ┆ 1        ┆ Carmencita  ┆ null   ┆ null     ┆ original    ┆ null        ┆ 1           │
│ tt0000001 ┆ 2        ┆ Carmencita  ┆ DE     ┆ null     ┆ null        ┆ literal     ┆ 0           │
│           ┆          ┆             ┆        ┆          ┆             ┆ title       ┆             │
│ tt0000001 ┆ 3        ┆ Carmencita  ┆ US     ┆ null  

In [ ]:
print("=== NULL rates ===")
akas_nulls = null_rates(akas)
print(akas_nulls)
akas_nulls.write_csv(OUT / "04_akas_null_rates.csv")

=== NULL rates ===
shape: (8, 3)
┌─────────────────┬──────────┬──────────┐
│ column          ┆ null     ┆ null_pct │
│ ---             ┆ ---      ┆ ---      │
│ str             ┆ i64      ┆ f64      │
╞═════════════════╪══════════╪══════════╡
│ titleId         ┆ 0        ┆ 0.0      │
│ ordering        ┆ 0        ┆ 0.0      │
│ title           ┆ 0        ┆ 0.0      │
│ region          ┆ 12530043 ┆ 21.98    │
│ language        ┆ 18694836 ┆ 32.79    │
│ types           ┆ 38758327 ┆ 67.98    │
│ attributes      ┆ 56703281 ┆ 99.46    │
│ isOriginalTitle ┆ 0        ┆ 0.0      │
└─────────────────┴──────────┴──────────┘


In [ ]:
print("=== Top 20 regions ===")
top_regions = (
    akas.filter(pl.col("region").is_not_null())
    .group_by("region")
    .len()
    .sort("len", descending=True)
    .head(20)
    .collect()
)
print(top_regions)

=== Top 20 regions ===
shape: (20, 2)
┌────────┬─────────┐
│ region ┆ len     │
│ ---    ┆ ---     │
│ str    ┆ u32     │
╞════════╪═════════╡
│ IN     ┆ 5567016 │
│ DE     ┆ 5499978 │
│ JP     ┆ 5476873 │
│ FR     ┆ 5457327 │
│ ES     ┆ 5371579 │
│ IT     ┆ 5342652 │
│ PT     ┆ 5249918 │
│ US     ┆ 1799836 │
│ GB     ┆ 654896  │
│ CA     ┆ 456352  │
│ AU     ┆ 352971  │
│ ZA     ┆ 241805  │
│ IE     ┆ 239546  │
│ NZ     ┆ 235457  │
│ XWW    ┆ 231482  │
│ BR     ┆ 144057  │
│ RU     ┆ 130959  │
│ MX     ┆ 124137  │
│ PL     ┆ 110195  │
│ GR     ┆ 100881  │
└────────┴─────────┘


In [ ]:
print("=== Distinct region count ===")
n_regions = (
    akas.filter(pl.col("region").is_not_null())
    .select(pl.col("region").n_unique())
    .collect()
    .item()
)
print(f"Distinct region codes: {n_regions}")

=== Distinct region count ===
Distinct region codes: 251


In [ ]:
print("=== Top 20 languages ===")
top_langs = (
    akas.filter(pl.col("language").is_not_null())
    .group_by("language")
    .len()
    .sort("len", descending=True)
    .head(20)
    .collect()
)
print(top_langs)

n_langs = (
    akas.filter(pl.col("language").is_not_null())
    .select(pl.col("language").n_unique())
    .collect()
    .item()
)
print(f"Distinct language codes: {n_langs}")

=== Top 20 languages ===
shape: (20, 2)
┌──────────┬─────────┐
│ language ┆ len     │
│ ---      ┆ ---     │
│ str      ┆ u32     │
╞══════════╪═════════╡
│ ja       ┆ 5314768 │
│ fr       ┆ 5252515 │
│ hi       ┆ 5216962 │
│ es       ┆ 5194783 │
│ de       ┆ 5181708 │
│ it       ┆ 5179167 │
│ pt       ┆ 5178986 │
│ en       ┆ 1552765 │
│ cmn      ┆ 52101   │
│ ru       ┆ 42306   │
│ tr       ┆ 41829   │
│ bg       ┆ 27919   │
│ sv       ┆ 11473   │
│ yue      ┆ 10788   │
│ he       ┆ 9122    │
│ ca       ┆ 8079    │
│ qbn      ┆ 8018    │
│ sr       ┆ 7818    │
│ fa       ┆ 4581    │
│ nl       ┆ 2198    │
└──────────┴─────────┘
Distinct language codes: 110


In [ ]:
print("=== AKA `types` — full distribution (IMDb says space-separated multiset) ===")
types_dist = (
    akas.filter(pl.col("types").is_not_null())
    .group_by("types")
    .len()
    .sort("len", descending=True)
    .head(40)
    .collect()
)
print(types_dist)
types_dist.write_csv(OUT / "05_akas_types_distribution.csv")

=== AKA `types` — full distribution (IMDb says space-separated multiset) ===
shape: (23, 2)
┌──────────────────────┬──────────┐
│ types                ┆ len      │
│ ---                  ┆ ---      │
│ str                  ┆ u32      │
╞══════════════════════╪══════════╡
│ original             ┆ 12474473 │
│ imdbDisplay          ┆ 5469788  │
│ alternative          ┆ 162349   │
│ working              ┆ 61144    │
│ video                ┆ 22984    │
│ dvd                  ┆ 22592    │
│ festival             ┆ 22123    │
│ tv                   ┆ 19752    │
│ imdbDisplaydvd      ┆ 207      │
│ imdbDisplaytv       ┆ 70       │
│ imdbDisplayfestival ┆ 60       │
│ imdbDisplayworking  ┆ 27       │
│ imdbDisplayvideo    ┆ 26       │
│ alternativetv       ┆ 9        │
│ workingvideo        ┆ 5        │
│ dvdalternative      ┆ 5        │
│ workingalternative  ┆ 5        │
│ tvvideo             ┆ 5        │
│ workingtv           ┆ 4        │
│ workingfestival     ┆ 3        │
│ dvdvi

In [ ]:
print("=== AKAs per title (cardinality at scale) ===")
akas_per_title = (
    akas.group_by("titleId")
    .len()
    .select(pl.col("len").alias("akas"))
)
stats = akas_per_title.select([
    pl.col("akas").min().alias("min"),
    pl.col("akas").max().alias("max"),
    pl.col("akas").mean().alias("mean"),
    pl.col("akas").median().alias("median"),
    pl.col("akas").quantile(0.95).alias("p95"),
    pl.col("akas").quantile(0.99).alias("p99"),
]).collect()
print(stats)

=== AKAs per title (cardinality at scale) ===
shape: (1, 6)
┌─────┬─────┬──────────┬────────┬─────┬──────┐
│ min ┆ max ┆ mean     ┆ median ┆ p95 ┆ p99  │
│ --- ┆ --- ┆ ---      ┆ ---    ┆ --- ┆ ---  │
│ u32 ┆ u32 ┆ f64      ┆ f64    ┆ f64 ┆ f64  │
╞═════╪═════╪══════════╪════════╪═════╪══════╡
│ 1   ┆ 313 ┆ 4.570449 ┆ 2.0    ┆ 8.0 ┆ 12.0 │
└─────┴─────┴──────────┴────────┴─────┴──────┘


## 3. `title.crew.tsv.gz` — directors + writers (CSV nconst lists)

In [ ]:
crew = pl.scan_csv(
    RAW / "title.crew.tsv.gz",
    separator="\t",
    null_values=NULL,
    quote_char=None,
)
crew_total = crew.select(pl.len()).collect().item()
print(f"Total rows: {crew_total:,}")

Total rows: 12,478,987


In [ ]:
print("=== First 10 rows ===")
print(crew.head(10).collect())

=== First 10 rows ===
shape: (10, 3)
┌───────────┬─────────────────────┬───────────┐
│ tconst    ┆ directors           ┆ writers   │
│ ---       ┆ ---                 ┆ ---       │
│ str       ┆ str                 ┆ str       │
╞═══════════╪═════════════════════╪═══════════╡
│ tt0000001 ┆ nm0005690           ┆ null      │
│ tt0000002 ┆ nm0721526           ┆ null      │
│ tt0000003 ┆ nm0721526           ┆ nm0721526 │
│ tt0000004 ┆ nm0721526           ┆ null      │
│ tt0000005 ┆ nm0005690           ┆ null      │
│ tt0000006 ┆ nm0005690           ┆ null      │
│ tt0000007 ┆ nm0005690,nm0374658 ┆ null      │
│ tt0000008 ┆ nm0005690           ┆ null      │
│ tt0000009 ┆ nm0085156           ┆ nm0085156 │
│ tt0000010 ┆ nm0525910           ┆ null      │
└───────────┴─────────────────────┴───────────┘


In [ ]:
print("=== NULL rates ===")
crew_nulls = null_rates(crew)
print(crew_nulls)
crew_nulls.write_csv(OUT / "06_crew_null_rates.csv")

=== NULL rates ===
shape: (3, 3)
┌───────────┬─────────┬──────────┐
│ column    ┆ null    ┆ null_pct │
│ ---       ┆ ---     ┆ ---      │
│ str       ┆ i64     ┆ f64      │
╞═══════════╪═════════╪══════════╡
│ tconst    ┆ 0       ┆ 0.0      │
│ directors ┆ 5508482 ┆ 44.14    │
│ writers   ┆ 6126697 ┆ 49.1     │
└───────────┴─────────┴──────────┘


In [ ]:
print("=== Directors per title (CSV cardinality) ===")
dir_card = csv_cardinality(crew.filter(pl.col("directors").is_not_null()), "directors")
# show distribution buckets
dir_buckets = (
    dir_card.with_columns(
        pl.when(pl.col("items_per_row") == 1).then(pl.lit("1"))
        .when(pl.col("items_per_row") <= 5).then(pl.lit("2-5"))
        .when(pl.col("items_per_row") <= 20).then(pl.lit("6-20"))
        .when(pl.col("items_per_row") <= 100).then(pl.lit("21-100"))
        .otherwise(pl.lit("100+"))
        .alias("bucket")
    )
    .group_by("bucket")
    .agg(pl.col("len").sum())
    .sort("bucket")
)
print(dir_buckets)
print(f"Max directors on a single tconst: {dir_card['items_per_row'].max()}")

=== Directors per title (CSV cardinality) ===
shape: (5, 2)
┌────────┬─────────┐
│ bucket ┆ len     │
│ ---    ┆ ---     │
│ str    ┆ u32     │
╞════════╪═════════╡
│ 1      ┆ 5581712 │
│ 100+   ┆ 86      │
│ 2-5    ┆ 1340642 │
│ 21-100 ┆ 2211    │
│ 6-20   ┆ 45854   │
└────────┴─────────┘
Max directors on a single tconst: 528


In [ ]:
print("=== Writers per title (CSV cardinality) ===")
wri_card = csv_cardinality(crew.filter(pl.col("writers").is_not_null()), "writers")
print(f"Max writers on a single tconst: {wri_card['items_per_row'].max()}")

wri_buckets = (
    wri_card.with_columns(
        pl.when(pl.col("items_per_row") == 1).then(pl.lit("1"))
        .when(pl.col("items_per_row") <= 5).then(pl.lit("2-5"))
        .when(pl.col("items_per_row") <= 20).then(pl.lit("6-20"))
        .when(pl.col("items_per_row") <= 100).then(pl.lit("21-100"))
        .otherwise(pl.lit("100+"))
        .alias("bucket")
    )
    .group_by("bucket")
    .agg(pl.col("len").sum())
    .sort("bucket")
)
print(wri_buckets)

=== Writers per title (CSV cardinality) ===
Max writers on a single tconst: 1393
shape: (5, 2)
┌────────┬─────────┐
│ bucket ┆ len     │
│ ---    ┆ ---     │
│ str    ┆ u32     │
╞════════╪═════════╡
│ 1      ┆ 3116094 │
│ 100+   ┆ 266     │
│ 2-5    ┆ 2772751 │
│ 21-100 ┆ 7393    │
│ 6-20   ┆ 455786  │
└────────┴─────────┘


## 4. `title.episode.tsv.gz`

In [ ]:
episode = pl.scan_csv(
    RAW / "title.episode.tsv.gz",
    separator="\t",
    null_values=NULL,
    schema_overrides={"seasonNumber": pl.Int32, "episodeNumber": pl.Int32},
    quote_char=None,
)
ep_total = episode.select(pl.len()).collect().item()
print(f"Total rows: {ep_total:,}")

Total rows: 9,637,883


In [ ]:
print("=== First 10 rows ===")
print(episode.head(10).collect())

=== First 10 rows ===
shape: (10, 4)
┌───────────┬──────────────┬──────────────┬───────────────┐
│ tconst    ┆ parentTconst ┆ seasonNumber ┆ episodeNumber │
│ ---       ┆ ---          ┆ ---          ┆ ---           │
│ str       ┆ str          ┆ i32          ┆ i32           │
╞═══════════╪══════════════╪══════════════╪═══════════════╡
│ tt0031458 ┆ tt32857063   ┆ null         ┆ null          │
│ tt0041951 ┆ tt0041038    ┆ 1            ┆ 9             │
│ tt0042816 ┆ tt0989125    ┆ 1            ┆ 17            │
│ tt0042889 ┆ tt0989125    ┆ null         ┆ null          │
│ tt0043426 ┆ tt0040051    ┆ 3            ┆ 42            │
│ tt0043631 ┆ tt0989125    ┆ 2            ┆ 16            │
│ tt0043693 ┆ tt0989125    ┆ 2            ┆ 8             │
│ tt0043710 ┆ tt0989125    ┆ 3            ┆ 3             │
│ tt0044093 ┆ tt0959862    ┆ 1            ┆ 6             │
│ tt0044668 ┆ tt0044243    ┆ 2            ┆ 16            │
└───────────┴──────────────┴──────────────┴───────────────┘


In [ ]:
print("=== NULL rates ===")
print(null_rates(episode))

=== NULL rates ===
shape: (4, 3)
┌───────────────┬─────────┬──────────┐
│ column        ┆ null    ┆ null_pct │
│ ---           ┆ ---     ┆ ---      │
│ str           ┆ i64     ┆ f64      │
╞═══════════════╪═════════╪══════════╡
│ tconst        ┆ 0       ┆ 0.0      │
│ parentTconst  ┆ 0       ┆ 0.0      │
│ seasonNumber  ┆ 1994519 ┆ 20.69    │
│ episodeNumber ┆ 1994519 ┆ 20.69    │
└───────────────┴─────────┴──────────┘


In [ ]:
print("=== Episodes per parent series (max + percentiles) ===")
ep_per_series = (
    episode.group_by("parentTconst")
    .len()
    .select(pl.col("len").alias("episodes"))
)
print(ep_per_series.select([
    pl.col("episodes").min().alias("min"),
    pl.col("episodes").max().alias("max"),
    pl.col("episodes").mean().alias("mean"),
    pl.col("episodes").median().alias("median"),
    pl.col("episodes").quantile(0.95).alias("p95"),
    pl.col("episodes").quantile(0.99).alias("p99"),
]).collect())

=== Episodes per parent series (max + percentiles) ===
shape: (1, 6)
┌─────┬───────┬───────────┬────────┬───────┬───────┐
│ min ┆ max   ┆ mean      ┆ median ┆ p95   ┆ p99   │
│ --- ┆ ---   ┆ ---       ┆ ---    ┆ ---   ┆ ---   │
│ u32 ┆ u32   ┆ f64       ┆ f64    ┆ f64   ┆ f64   │
╞═════╪═══════╪═══════════╪════════╪═══════╪═══════╡
│ 1   ┆ 26422 ┆ 40.861169 ┆ 8.0    ┆ 135.0 ┆ 574.0 │
└─────┴───────┴───────────┴────────┴───────┴───────┘


In [ ]:
print("=== Distinct parent series count ===")
n_series = episode.select(pl.col("parentTconst").n_unique()).collect().item()
print(f"Distinct parents: {n_series:,}")

=== Distinct parent series count ===
Distinct parents: 235,869


## 5. `title.principals.tsv.gz` — per-credit cast/crew (largest file: ~85M rows)

In [ ]:
principals = pl.scan_csv(
    RAW / "title.principals.tsv.gz",
    separator="\t",
    null_values=NULL,
    schema_overrides={"ordering": pl.Int32},
    quote_char=None,
)
prin_total = principals.select(pl.len()).collect().item()
print(f"Total rows: {prin_total:,}")

Total rows: 99,275,168


In [ ]:
print("=== First 10 rows ===")
print(principals.head(10).collect())

=== First 10 rows ===
shape: (10, 6)
┌───────────┬──────────┬───────────┬─────────────────┬─────────────────────────┬────────────┐
│ tconst    ┆ ordering ┆ nconst    ┆ category        ┆ job                     ┆ characters │
│ ---       ┆ ---      ┆ ---       ┆ ---             ┆ ---                     ┆ ---        │
│ str       ┆ i32      ┆ str       ┆ str             ┆ str                     ┆ str        │
╞═══════════╪══════════╪═══════════╪═════════════════╪═════════════════════════╪════════════╡
│ tt0000001 ┆ 1        ┆ nm1588970 ┆ self            ┆ null                    ┆ ["Self"]   │
│ tt0000001 ┆ 2        ┆ nm0005690 ┆ director        ┆ null                    ┆ null       │
│ tt0000001 ┆ 3        ┆ nm0005690 ┆ producer        ┆ producer                ┆ null       │
│ tt0000001 ┆ 4        ┆ nm0374658 ┆ cinematographer ┆ director of photography ┆ null       │
│ tt0000002 ┆ 1        ┆ nm0721526 ┆ director        ┆ null                    ┆ null       │
│ tt0000002 ┆ 2        

In [ ]:
# count unique value per column
for c in principals.schema.names():
    n_unique = principals.select(pl.col(c).n_unique()).collect().item()
    print(f"Column '{c}': {n_unique} unique values")

=== NULL rates ===


In [ ]:
print("=== NULL rates ===")
prin_nulls = null_rates(principals)
print(prin_nulls)
prin_nulls.write_csv(OUT / "07_principals_null_rates.csv")

=== NULL rates ===
shape: (6, 3)
┌────────────┬──────────┬──────────┐
│ column     ┆ null     ┆ null_pct │
│ ---        ┆ ---      ┆ ---      │
│ str        ┆ i64      ┆ f64      │
╞════════════╪══════════╪══════════╡
│ tconst     ┆ 0        ┆ 0.0      │
│ ordering   ┆ 0        ┆ 0.0      │
│ nconst     ┆ 0        ┆ 0.0      │
│ category   ┆ 0        ┆ 0.0      │
│ job        ┆ 80286734 ┆ 80.87    │
│ characters ┆ 50825606 ┆ 51.2     │
└────────────┴──────────┴──────────┘


In [ ]:
print("=== category distribution (full IMDb) ===")
cat_dist = (
    principals.group_by("category")
    .len()
    .sort("len", descending=True)
    .collect()
)
print(cat_dist)
cat_dist.write_csv(OUT / "08_principals_category_distribution.csv")

=== category distribution (full IMDb) ===
shape: (13, 2)
┌─────────────────────┬──────────┐
│ category            ┆ len      │
│ ---                 ┆ ---      │
│ str                 ┆ u32      │
╞═════════════════════╪══════════╡
│ actor               ┆ 23457533 │
│ actress             ┆ 17732782 │
│ self                ┆ 14971060 │
│ writer              ┆ 11888962 │
│ director            ┆ 8451762  │
│ producer            ┆ 7379486  │
│ editor              ┆ 5218239  │
│ cinematographer     ┆ 4010468  │
│ composer            ┆ 3191150  │
│ production_designer ┆ 1182366  │
│ casting_director    ┆ 1144162  │
│ archive_footage     ┆ 633910   │
│ archive_sound       ┆ 13288    │
└─────────────────────┴──────────┘


In [ ]:
print("=== Distinct `job` values count ===")
n_jobs = principals.filter(pl.col("job").is_not_null()).select(pl.col("job").n_unique()).collect().item()
print(f"Distinct job free-text values: {n_jobs:,}")

=== Distinct `job` values count ===
Distinct job free-text values: 47,125


In [ ]:
print("=== Top 30 `job` values ===")
top_jobs = (
    principals.filter(pl.col("job").is_not_null())
    .group_by("job")
    .len()
    .sort("len", descending=True)
    .head(30)
    .collect()
)
print(top_jobs)
top_jobs.write_csv(OUT / "09_principals_top_jobs.csv")

=== Top 30 `job` values ===
shape: (30, 2)
┌─────────────────────────┬─────────┐
│ job                     ┆ len     │
│ ---                     ┆ ---     │
│ str                     ┆ u32     │
╞═════════════════════════╪═════════╡
│ producer                ┆ 7196543 │
│ writer                  ┆ 1838828 │
│ director                ┆ 1004819 │
│ written by              ┆ 784104  │
│ creator                 ┆ 679437  │
│ editor                  ┆ 670069  │
│ composer                ┆ 646829  │
│ created by              ┆ 637089  │
│ director of photography ┆ 590254  │
│ cinematographer         ┆ 431286  │
│ screenplay              ┆ 385228  │
│ story                   ┆ 300618  │
│ head writer             ┆ 267541  │
│ casting_director        ┆ 171024  │
│ executive producer      ┆ 159472  │
│ dialogue                ┆ 152296  │
│ production_designer     ┆ 147229  │
│ adaptation              ┆ 134439  │
│ creative director       ┆ 82691   │
│ series director         ┆ 81238   │
│ story

In [ ]:
print("=== Characters JSON cardinality (count of '\",\"' separators per non-null value) ===")
# An entry like ["A","B","C"] has two `","` separators → 3 elements.
chars_card = (
    principals.filter(pl.col("characters").is_not_null())
    .select(
        (pl.col("characters").str.count_matches('","') + 1).alias("char_count")
    )
    .group_by("char_count")
    .len()
    .sort("char_count")
    .collect()
)
print(chars_card)
chars_card.write_csv(OUT / "10_characters_cardinality.csv")
print(f"Max characters in a single credit: {chars_card['char_count'].max()}")

=== Characters JSON cardinality (count of '","' separators per non-null value) ===
shape: (2, 2)
┌────────────┬──────────┐
│ char_count ┆ len      │
│ ---        ┆ ---      │
│ u32        ┆ u32      │
╞════════════╪══════════╡
│ 1          ┆ 48449556 │
│ 2          ┆ 6        │
└────────────┴──────────┘
Max characters in a single credit: 2


In [ ]:
print("=== Principals per title (mean / max / percentiles, full IMDb) ===")
prin_per_title = (
    principals.group_by("tconst")
    .len()
    .select(pl.col("len").alias("principals"))
)
print(prin_per_title.select([
    pl.col("principals").min().alias("min"),
    pl.col("principals").max().alias("max"),
    pl.col("principals").mean().alias("mean"),
    pl.col("principals").median().alias("median"),
    pl.col("principals").quantile(0.95).alias("p95"),
    pl.col("principals").quantile(0.99).alias("p99"),
]).collect())

=== Principals per title (mean / max / percentiles, full IMDb) ===
shape: (1, 6)
┌─────┬─────┬──────────┬────────┬──────┬──────┐
│ min ┆ max ┆ mean     ┆ median ┆ p95  ┆ p99  │
│ --- ┆ --- ┆ ---      ┆ ---    ┆ ---  ┆ ---  │
│ u32 ┆ u32 ┆ f64      ┆ f64    ┆ f64  ┆ f64  │
╞═════╪═════╪══════════╪════════╪══════╪══════╡
│ 1   ┆ 75  ┆ 8.796415 ┆ 8.0    ┆ 20.0 ┆ 25.0 │
└─────┴─────┴──────────┴────────┴──────┴──────┘


In [ ]:
print("=== Credits per person (mean / max / percentiles, full IMDb) ===")
credits_per_person = (
    principals.group_by("nconst")
    .len()
    .select(pl.col("len").alias("credits"))
)
print(credits_per_person.select([
    pl.col("credits").min().alias("min"),
    pl.col("credits").max().alias("max"),
    pl.col("credits").mean().alias("mean"),
    pl.col("credits").median().alias("median"),
    pl.col("credits").quantile(0.95).alias("p95"),
    pl.col("credits").quantile(0.99).alias("p99"),
]).collect())

=== Credits per person (mean / max / percentiles, full IMDb) ===
shape: (1, 6)
┌─────┬───────┬───────────┬────────┬──────┬───────┐
│ min ┆ max   ┆ mean      ┆ median ┆ p95  ┆ p99   │
│ --- ┆ ---   ┆ ---       ┆ ---    ┆ ---  ┆ ---   │
│ u32 ┆ u32   ┆ f64       ┆ f64    ┆ f64  ┆ f64   │
╞═════╪═══════╪═══════════╪════════╪══════╪═══════╡
│ 1   ┆ 39463 ┆ 14.055879 ┆ 1.0    ┆ 39.0 ┆ 225.0 │
└─────┴───────┴───────────┴────────┴──────┴───────┘


## 6. `title.ratings.tsv.gz`

In [ ]:
ratings = pl.scan_csv(
    RAW / "title.ratings.tsv.gz",
    separator="\t",
    null_values=NULL,
    schema_overrides={"averageRating": pl.Float32, "numVotes": pl.Int64},
    quote_char=None,
)
print(f"Total rows: {ratings.select(pl.len()).collect().item():,}")

Total rows: 1,668,663


In [ ]:
print("=== First 10 rows ===")
print(ratings.head(10).collect())

=== First 10 rows ===
shape: (10, 3)
┌───────────┬───────────────┬──────────┐
│ tconst    ┆ averageRating ┆ numVotes │
│ ---       ┆ ---           ┆ ---      │
│ str       ┆ f32           ┆ i64      │
╞═══════════╪═══════════════╪══════════╡
│ tt0000001 ┆ 5.7           ┆ 2213     │
│ tt0000002 ┆ 5.5           ┆ 317      │
│ tt0000003 ┆ 6.4           ┆ 2327     │
│ tt0000004 ┆ 5.1           ┆ 199      │
│ tt0000005 ┆ 6.2           ┆ 3047     │
│ tt0000006 ┆ 5.1           ┆ 228      │
│ tt0000007 ┆ 5.3           ┆ 949      │
│ tt0000008 ┆ 5.4           ┆ 2394     │
│ tt0000009 ┆ 5.3           ┆ 240      │
│ tt0000010 ┆ 6.8           ┆ 8320     │
└───────────┴───────────────┴──────────┘


In [ ]:
print("=== NULL rates ===")
print(null_rates(ratings))
print("=== Stats ===")
print(ratings.select([
    pl.col("averageRating").min().alias("rat_min"),
    pl.col("averageRating").max().alias("rat_max"),
    pl.col("averageRating").mean().alias("rat_mean"),
    pl.col("numVotes").min().alias("votes_min"),
    pl.col("numVotes").max().alias("votes_max"),
    pl.col("numVotes").mean().alias("votes_mean"),
    pl.col("numVotes").median().alias("votes_median"),
]).collect())

=== NULL rates ===
shape: (3, 3)
┌───────────────┬──────┬──────────┐
│ column        ┆ null ┆ null_pct │
│ ---           ┆ ---  ┆ ---      │
│ str           ┆ i64  ┆ f64      │
╞═══════════════╪══════╪══════════╡
│ tconst        ┆ 0    ┆ 0.0      │
│ averageRating ┆ 0    ┆ 0.0      │
│ numVotes      ┆ 0    ┆ 0.0      │
└───────────────┴──────┴──────────┘
=== Stats ===
shape: (1, 7)
┌─────────┬─────────┬──────────┬───────────┬───────────┬─────────────┬──────────────┐
│ rat_min ┆ rat_max ┆ rat_mean ┆ votes_min ┆ votes_max ┆ votes_mean  ┆ votes_median │
│ ---     ┆ ---     ┆ ---      ┆ ---       ┆ ---       ┆ ---         ┆ ---          │
│ f32     ┆ f32     ┆ f32      ┆ i64       ┆ i64       ┆ f64         ┆ f64          │
╞═════════╪═════════╪══════════╪═══════════╪═══════════╪═════════════╪══════════════╡
│ 1.0     ┆ 10.0    ┆ 6.960398 ┆ 5         ┆ 3185031   ┆ 1036.960987 ┆ 27.0         │
└─────────┴─────────┴──────────┴───────────┴───────────┴─────────────┴──────────────┘


## 7. `name.basics.tsv.gz`

In [ ]:
names = pl.scan_csv(
    RAW / "name.basics.tsv.gz",
    separator="\t",
    null_values=NULL,
    schema_overrides={"birthYear": pl.Int32, "deathYear": pl.Int32},
    quote_char=None,
)
names_total = names.select(pl.len()).collect().item()
print(f"Total rows: {names_total:,}")

Total rows: 15,297,466


In [ ]:
print("=== First 10 rows ===")
print(names.head(10).collect())

=== First 10 rows ===
shape: (10, 6)
┌───────────┬─────────────────┬───────────┬───────────┬──────────────────────┬─────────────────────┐
│ nconst    ┆ primaryName     ┆ birthYear ┆ deathYear ┆ primaryProfession    ┆ knownForTitles      │
│ ---       ┆ ---             ┆ ---       ┆ ---       ┆ ---                  ┆ ---                 │
│ str       ┆ str             ┆ i32       ┆ i32       ┆ str                  ┆ str                 │
╞═══════════╪═════════════════╪═══════════╪═══════════╪══════════════════════╪═════════════════════╡
│ nm0000001 ┆ Fred Astaire    ┆ 1899      ┆ 1987      ┆ actor,miscellaneous, ┆ tt0072308,tt0050419 │
│           ┆                 ┆           ┆           ┆ producer             ┆ ,tt0027125,…        │
│ nm0000002 ┆ Lauren Bacall   ┆ 1924      ┆ 2014      ┆ actress,miscellaneou ┆ tt0037382,tt0075213 │
│           ┆                 ┆           ┆           ┆ s,soundtra…          ┆ ,tt0038355,…        │
│ nm0000003 ┆ Brigitte Bardot ┆ 1934      ┆ 2025      

In [ ]:
print("=== NULL rates ===")
names_nulls = null_rates(names)
print(names_nulls)
names_nulls.write_csv(OUT / "11_names_null_rates.csv")

=== NULL rates ===
shape: (6, 3)
┌───────────────────┬──────────┬──────────┐
│ column            ┆ null     ┆ null_pct │
│ ---               ┆ ---      ┆ ---      │
│ str               ┆ i64      ┆ f64      │
╞═══════════════════╪══════════╪══════════╡
│ nconst            ┆ 0        ┆ 0.0      │
│ primaryName       ┆ 67       ┆ 0.0      │
│ birthYear         ┆ 14625420 ┆ 95.61    │
│ deathYear         ┆ 15039310 ┆ 98.31    │
│ primaryProfession ┆ 3078253  ┆ 20.12    │
│ knownForTitles    ┆ 1826655  ┆ 11.94    │
└───────────────────┴──────────┴──────────┘


In [ ]:
print("=== Distinct primaryProfession values across full IMDb ===")
prof_distinct = (
    names.filter(pl.col("primaryProfession").is_not_null())
    .select(pl.col("primaryProfession").str.split(",").alias("p"))
    .explode("p")
    .group_by("p")
    .len()
    .sort("len", descending=True)
    .collect()
)
print(f"Distinct professions: {prof_distinct.height}")
print(prof_distinct)
prof_distinct.write_csv(OUT / "12_distinct_professions.csv")

=== Distinct primaryProfession values across full IMDb ===
Distinct professions: 46
shape: (46, 2)
┌───────────────────────┬─────────┐
│ p                     ┆ len     │
│ ---                   ┆ ---     │
│ str                   ┆ u32     │
╞═══════════════════════╪═════════╡
│ actor                 ┆ 3499572 │
│ actress               ┆ 2112958 │
│ producer              ┆ 1342015 │
│ miscellaneous         ┆ 1219417 │
│ writer                ┆ 1007414 │
│ camera_department     ┆ 894191  │
│ director              ┆ 826203  │
│ production_department ┆ 601057  │
│ art_department        ┆ 476486  │
│ cinematographer       ┆ 443369  │
│ sound_department      ┆ 433888  │
│ editor                ┆ 405133  │
│ composer              ┆ 384399  │
│ music_department      ┆ 320196  │
│ assistant_director    ┆ 290913  │
│ visual_effects        ┆ 274772  │
│ make_up_department    ┆ 253568  │
│ animation_department  ┆ 248773  │
│ archive_footage       ┆ 231349  │
│ production_manager    ┆ 208671  │
│

In [ ]:
print("=== primaryProfession cardinality (CSV items per row) ===")
prof_card = csv_cardinality(names.filter(pl.col("primaryProfession").is_not_null()), "primaryProfession")
print(prof_card)
print(f"Max primaryProfession items: {prof_card['items_per_row'].max()}")

=== primaryProfession cardinality (CSV items per row) ===
shape: (3, 2)
┌───────────────┬─────────┐
│ items_per_row ┆ len     │
│ ---           ┆ ---     │
│ u32           ┆ u32     │
╞═══════════════╪═════════╡
│ 1             ┆ 9026569 │
│ 2             ┆ 1576312 │
│ 3             ┆ 1616332 │
└───────────────┴─────────┘
Max primaryProfession items: 3


In [ ]:
print("=== knownForTitles cardinality ===")
kft_card = csv_cardinality(names.filter(pl.col("knownForTitles").is_not_null()), "knownForTitles")
print(kft_card)
print(f"Max knownForTitles items: {kft_card['items_per_row'].max()}")

=== knownForTitles cardinality ===
shape: (4, 2)
┌───────────────┬─────────┐
│ items_per_row ┆ len     │
│ ---           ┆ ---     │
│ u32           ┆ u32     │
╞═══════════════╪═════════╡
│ 1             ┆ 8276605 │
│ 2             ┆ 1718738 │
│ 3             ┆ 781225  │
│ 4             ┆ 2694243 │
└───────────────┴─────────┘
Max knownForTitles items: 4


## 8. Cross-file integrity at full scale

All checks are lazy — polars streams them efficiently.

In [ ]:
print("=== title.principals.tconst → title.basics.tconst ===")
basics_tconsts = basics.select("tconst").rename({"tconst": "tconst_b"})
prin_tconsts = principals.select("tconst").unique()

orphan_prin_titles = (
    prin_tconsts.join(basics_tconsts, left_on="tconst", right_on="tconst_b", how="anti")
    .select(pl.len())
    .collect()
    .item()
)
total_prin_titles = prin_tconsts.select(pl.len()).collect().item()
print(f"  principals references {total_prin_titles:,} distinct tconsts; "
      f"{orphan_prin_titles:,} are missing from title.basics ({100*orphan_prin_titles/max(total_prin_titles,1):.2f}%)")

=== title.principals.tconst → title.basics.tconst ===
  principals references 11,285,867 distinct tconsts; 0 are missing from title.basics (0.00%)


In [ ]:
print("=== title.principals.nconst → name.basics.nconst ===")
names_nconsts = names.select("nconst").rename({"nconst": "nconst_n"})
prin_nconsts = principals.select("nconst").unique()
orphan_prin_persons = (
    prin_nconsts.join(names_nconsts, left_on="nconst", right_on="nconst_n", how="anti")
    .select(pl.len())
    .collect()
    .item()
)
total_prin_persons = prin_nconsts.select(pl.len()).collect().item()
print(f"  principals references {total_prin_persons:,} distinct nconsts; "
      f"{orphan_prin_persons:,} are missing from name.basics "
      f"({100*orphan_prin_persons/max(total_prin_persons,1):.2f}%)")

=== title.principals.nconst → name.basics.nconst ===
  principals references 7,062,893 distinct nconsts; 1,667 are missing from name.basics (0.02%)


In [ ]:
print("=== title.episode.parentTconst → title.basics.tconst ===")
ep_parents = episode.select("parentTconst").unique()
orphan_parents = (
    ep_parents.join(basics_tconsts, left_on="parentTconst", right_on="tconst_b", how="anti")
    .select(pl.len())
    .collect()
    .item()
)
total_parents = ep_parents.select(pl.len()).collect().item()
print(f"  episode references {total_parents:,} distinct parents; "
      f"{orphan_parents:,} are missing from title.basics "
      f"({100*orphan_parents/max(total_parents,1):.2f}%)")

=== title.episode.parentTconst → title.basics.tconst ===
  episode references 235,869 distinct parents; 2 are missing from title.basics (0.00%)


In [ ]:
print("=== title.akas.titleId → title.basics.tconst ===")
akas_titles = akas.select("titleId").unique()
orphan_akas = (
    akas_titles.join(basics_tconsts, left_on="titleId", right_on="tconst_b", how="anti")
    .select(pl.len())
    .collect()
    .item()
)
total_akas_titles = akas_titles.select(pl.len()).collect().item()
print(f"  akas references {total_akas_titles:,} distinct titleIds; "
      f"{orphan_akas:,} are missing from title.basics "
      f"({100*orphan_akas/max(total_akas_titles,1):.2f}%)")

=== title.akas.titleId → title.basics.tconst ===
  akas references 12,474,477 distinct titleIds; 56 are missing from title.basics (0.00%)


## Summary

Save a compact JSON of the headline numbers for cross-reference from the markdown report.

In [ ]:
summary = {
    "row_counts": {
        "title.basics": basics_total,
        "title.akas": akas_total,
        "title.crew": crew_total,
        "title.episode": ep_total,
        "title.principals": prin_total,
        "name.basics": names_total,
    },
    "fk_orphans": {
        "principals.tconst_orphans": orphan_prin_titles,
        "principals.nconst_orphans": orphan_prin_persons,
        "episode.parent_orphans": orphan_parents,
        "akas.titleId_orphans": orphan_akas,
    },
    "max_cardinalities": {
        "directors_per_tconst": int(dir_card["items_per_row"].max()),
        "writers_per_tconst": int(wri_card["items_per_row"].max()),
        "characters_per_credit": int(chars_card["char_count"].max()),
        "primaryProfession_per_nconst": int(prof_card["items_per_row"].max()),
        "knownForTitles_per_nconst": int(kft_card["items_per_row"].max()),
        "akas_per_title": int(stats["max"].item()),
    },
    "distinct_taxonomies": {
        "genres": genres_distinct.height,
        "professions": prof_distinct.height,
        "categories": int(cat_dist.height),
        "regions": int(n_regions),
        "languages": int(n_langs),
    },
}

with open(OUT / "summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print(f"\nWrote {OUT/'summary.json'} and 12 supporting CSVs to {OUT}/")

{
  "row_counts": {
    "title.basics": 12478987,
    "title.akas": 57013962,
    "title.crew": 12478987,
    "title.episode": 9637883,
    "title.principals": 99275168,
    "name.basics": 15297466
  },
  "fk_orphans": {
    "principals.tconst_orphans": 0,
    "principals.nconst_orphans": 1667,
    "episode.parent_orphans": 2,
    "akas.titleId_orphans": 56
  },
  "max_cardinalities": {
    "directors_per_tconst": 528,
    "writers_per_tconst": 1393,
    "characters_per_credit": 2,
    "primaryProfession_per_nconst": 3,
    "knownForTitles_per_nconst": 4,
    "akas_per_title": 313
  },
  "distinct_taxonomies": {
    "genres": 28,
    "professions": 46,
    "categories": 13,
    "regions": 251,
    "languages": 110
  }
}

Wrote /home/duvu/All/82_Master_Uliege/_Y2Q2/KRR/info9014-krr-project/notebooks/output/summary.json and 12 supporting CSVs to /home/duvu/All/82_Master_Uliege/_Y2Q2/KRR/info9014-krr-project/notebooks/output/
